### 06.03. Setup functions and indexes

In [ ]:
# Configure the Azure OpenAI connection used by the agent and the embedding model.
# We keep secrets in environment variables instead of embedding them directly in the notebook so
# the exercise is safer to share and easier to run in different environments.
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

import os
from dotenv import load_dotenv

# Load the API settings from a local .env file. Using override=True ensures each re-run picks up the
# latest credentials instead of stale values left behind in the active notebook kernel.
load_dotenv(override=True)
api_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# Create the LLM that will orchestrate function calls and reason over user requests.
# The model needs tool-calling support so it can decide when to fetch order data, return dates,
# support information, or policy details.
llm = AzureChatOpenAI(
    azure_deployment="gpt-5.4-mini",
    api_key=api_key,
    azure_endpoint=azure_endpoint,
    api_version=api_version,
)

# Build the embedding model used for semantic retrieval from support documentation.
# This allows the agent to search policy text by meaning instead of matching only exact keywords.
embed_model = AzureOpenAIEmbeddings(
    azure_deployment="text-embedding-ada-002",
    api_key=api_key,
    azure_endpoint=azure_endpoint,
    api_version=api_version,
)

In [ ]:
from typing import List
from langchain_pymupdf4llm import PyMuPDF4LLMLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

# ------------------------------------------------------------------
# Tool 1: Return the list of items in a given order
# ------------------------------------------------------------------
# This tool gives the agent a way to answer questions like "What did I buy in order 1001?"
# The function is intentionally simple and deterministic, which makes it a good example of a
# structured tool that the LLM can call when it decides the user needs order history.
def get_order_items(order_id: int) -> List[str]:
    """Return the list of items purchased for a given order ID."""

    order_items = {
        1001: ["Laptop", "Mouse"],
        1002: ["Keyboard", "HDMI Cable"],
        1003: ["Laptop", "Keyboard"],
    }
    if order_id in order_items:
        return order_items[order_id]
    else:
        return []

# ------------------------------------------------------------------
# Tool 2: Return the delivery date for an order
# ------------------------------------------------------------------
# This tool helps the agent answer timing questions such as when an order will arrive.
# It is a good example of a narrow, factual tool that reduces the chance of the model making up
# delivery information without consulting the data source.
def get_delivery_date(order_id: int) -> str:
    """Return the delivery date for a given order ID."""

    delivery_dates = {
        1001: "10-Jun",
        1002: "12-Jun",
        1003: "08-Jun",
    }
    if order_id in delivery_dates:
        return delivery_dates[order_id]
    else:
        return []

# ------------------------------------------------------------------
# Tool 3: Return the maximum return days for an item
# ------------------------------------------------------------------
# This tool adds business rules to the agent. Instead of guessing policy, the LLM can rely on a
# defined return window for each product category, which is exactly what an operational assistant
# should be doing when answering customer support questions.
def get_item_return_days(item: str) -> int:
    """Return the number of return days allowed for an item."""

    item_returns = {
        "Laptop": 30,
        "Mouse": 15,
        "Keyboard": 15,
        "HDMI Cable": 5,
    }
    if item in item_returns:
        return item_returns[item]
    else:
        # Default rule when an item is not explicitly mapped.
        return 45

# ------------------------------------------------------------------
# Tool 4: Vector database containing customer support policies and contacts
# ------------------------------------------------------------------
# This retrieval tool handles less structured support knowledge such as policies, contacts, and
# procedure notes. The PDF is not queried as raw text; instead, it is converted into searchable
# embeddings so the agent can match the meaning of a customer question to the most relevant policy
# passages, even when the wording is different.

# Step 1: Load the PDF pages.
# Why: the support document is a long-form file, and we want to read it page by page so we preserve
# the original structure and can later retrieve the most relevant sections.
support_documents = PyMuPDF4LLMLoader("Customer Service.pdf", mode="page").load()

# Step 2: Split the document into smaller chunks.
# Why: large PDFs are too long for a single retrieval unit. Chunking creates focused passages so the
# similarity search can match a question to the right section of policy text instead of returning a
# broad and noisy chunk.
splitter = RecursiveCharacterTextSplitter(chunk_size=1024)
support_chunks = splitter.split_documents(support_documents)

# Step 3: Create an in-memory vector store and embed each chunk.
# Why: the embedding model turns each chunk into a vector representation. This allows semantic
# similarity search: the question is embedded as well, and the system finds chunks with similar
# meaning instead of only exact keyword matches.
support_vectorstore = InMemoryVectorStore(embed_model)
support_vectorstore.add_documents(support_chunks)

# Step 4: Turn the vector store into a retriever.
# Why: a retriever is the interface the agent uses to ask, "Which support passages are most relevant
# to this customer question?" It returns the top matching chunks that can be passed back into the
# model as context for the final answer.
support_retriever = support_vectorstore.as_retriever()

### 06.04. Setup the Customer Service AI Agent

In [ ]:
from langchain_core.tools import tool

# Wrap the helper functions and the retrieval system as LangChain tools.
# The agent can then choose which tool to call dynamically based on the user request instead of
# having a single hard-coded script for every possible support question.
order_item_tool = tool(get_order_items)
delivery_date_tool = tool(get_delivery_date)
return_policy_tool = tool(get_item_return_days)

@tool
def support_tool(query: str) -> str:
    """Customer support policies and contact information."""
    # Retrieve the most relevant support passages for the user's question and return them as text.
    # This is important because customer support knowledge is often stored in documents rather than
    # in a small fixed dictionary, and semantic retrieval helps the agent find the right policy.
    docs = support_retriever.invoke(query)
    return "\n\n".join(d.page_content for d in docs)

In [ ]:
from langchain.agents import create_agent

# Build the customer service agent with all of the available tools.
# This is the orchestration layer: the model decides which tools to call and in what sequence,
# rather than the developer hard-coding a single path for every possible customer request.
agent = create_agent(
    model=llm,
    tools=[order_item_tool, delivery_date_tool, return_policy_tool, support_tool],
)

# Run the agent and print the tool calls as they happen.
# This is useful because it exposes the agent's decision path in real time: we can see exactly which
# tool was chosen, what data was retrieved, and how the final answer was formed. That makes it
# easier for students to understand the workflow and for developers to debug unexpected behavior.
def run_agent(question: str) -> str:
    print(f"Added user message to memory: {question}")
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    final_answer = None

    # Walk through the full conversation trace returned by the agent so we can inspect every message
    # in order. This gives us visibility into the tool-calling sequence and the final output.
    for message in result["messages"]:
        # If the model is choosing to call a tool, log the exact function name and arguments.
        # This explains what the agent is trying to fetch before it answers the user's question.
        if message.type == "ai" and message.tool_calls:
            for call in message.tool_calls:
                print("=== Calling Function ===")
                print(f"Calling function: {call['name']} with args: {call['args']}")
        # If the message is a tool result, print that output so students can see the raw data the
        # agent used to ground its answer. This is the "why" behind the final response.
        elif message.type == "tool":
            print("=== Function Output ===")
            print(message.content)
        # If the message is the final assistant response with no tool calls, capture it as the answer
        # to return to the user. This is the point where the agent has finished reasoning and is
        # presenting the final result.
        elif message.type == "ai" and not message.tool_calls:
            final_answer = message.content

    print("=== LLM Response ===")
    print(final_answer)
    return final_answer

### 06.05. Using the customer service Agent

In [ ]:
# Ask the agent about a return policy for a specific order.
# This is a good first example because it exercises the agent's ability to combine order lookup and
# policy reasoning in a single user request.
response = run_agent(
    "What is the return policy for order number 1001"
)

print("\n Final output : \n", response)

Added user message to memory: What is the return policy for order number 1001
=== Calling Function ===
Calling function: get_order_items with args: {'order_id': 1001}
=== Calling Function ===
Calling function: support_tool with args: {'query': 'return policy for order number 1001'}
=== Function Output ===
["Laptop", "Mouse"]
=== Function Output ===
Customer Service for customers is available Monday to Friday from 8 AM to 5 PM. Customers can contact customer service on 1-987-654-3210. Or email us at support@company.com
=== LLM Response ===
Order **1001** includes a **Laptop** and a **Mouse**.

I also checked the support policy info available:
- Customer service hours: **Monday to Friday, 8 AM to 5 PM**
- Phone: **1-987-654-3210**
- Email: **support@company.com**

I don’t have the specific return window for that order from the available policy data. If you want, I can check the return period for the **Laptop** and **Mouse** individually.

 Final output : 
 Order **1001** includes a **Lap

In [ ]:
# Ask a multi-step customer support question that requires multiple tool calls.
# This helps students see the real value of an agent: it can route a single complex query through
# several tools before producing a final answer that combines the results.
response = run_agent(
    "When is the delivery date and items shipped for order 1003 and how can I contact customer support?"
)

print("\n Final output : \n", response)

Added user message to memory: When is the delivery date and items shipped for order 1003 and how can I contact customer support?
=== Calling Function ===
Calling function: get_delivery_date with args: {'order_id': 1003}
=== Calling Function ===
Calling function: get_order_items with args: {'order_id': 1003}
=== Calling Function ===
Calling function: support_tool with args: {'query': 'How can I contact customer support?'}
=== Function Output ===
08-Jun
=== Function Output ===
["Laptop", "Keyboard"]
=== Function Output ===
Customer Service for customers is available Monday to Friday from 8 AM to 5 PM. Customers can contact customer service on 1-987-654-3210. Or email us at support@company.com
=== LLM Response ===
Order **1003**:
- **Delivery date:** **08-Jun**
- **Items shipped:** **Laptop** and **Keyboard**

**Customer support contact:**
- **Phone:** 1-987-654-3210
- **Email:** support@company.com
- **Hours:** Monday to Friday, 8 AM to 5 PM

 Final output : 
 Order **1003**:
- **Deliver

In [ ]:
# Test how the agent behaves when the order number does not exist.
# This is an important real-world scenario: the agent should not hallucinate data or pretend to know
# a missing order; it should explain that the information is unavailable or ask for a valid order ID.
response = run_agent(
    "What is the return policy for order number 1004"
)

print("\n Final output : \n", response)

Added user message to memory: What is the return policy for order number 1004
=== Calling Function ===
Calling function: get_order_items with args: {'order_id': 1004}
=== Calling Function ===
Calling function: support_tool with args: {'query': 'return policy for order number 1004'}
=== Function Output ===
[]
=== Function Output ===
Customer Service for customers is available Monday to Friday from 8 AM to 5 PM. Customers can contact customer service on 1-987-654-3210. Or email us at support@company.com
=== LLM Response ===
For order **1004**, I’m able to confirm customer support contact info, but I couldn’t retrieve the order items needed to determine the exact return policy for that order.

**Customer service:**
- **Phone:** 1-987-654-3210
- **Email:** support@company.com
- **Hours:** Monday to Friday, 8 AM to 5 PM

If you want, I can help figure out the return window if you can share the items in the order.

 Final output : 
 For order **1004**, I’m able to confirm customer support co

: 